# Initial KIN modelling

**Document Identifier:** `KIN - DRAFT - 260916 - Initial modelling` 

**Author:** Yeray Alcaraz Galván  

**Affiliation:** Department of Chemistry – Ångström Laboratory, Uppsala University  

**Project:** Master's Thesis (TFM) — Battery Multiscale Chemistry & Reactor Modeling  

**Academic Supervisors:** Prof. Peter Broqvist  

**Date:** September 16, 2026  

---


## 1. Summary

> ### 1. MACE-OMol Input Data
> $E_{\text{0K}}, \{\nu_i\}, I_A, I_B, I_C, M$

$$\Big\downarrow \text{\small Quasi-RRHO Statistical Mechanics}$$

> ### 2. Species Thermochemistry $f(T)$
> $H_i(T),\; S_i(T),\; G_i(T)$

$$\Big\downarrow \text{\small Stoichiometric Matrix } \mathbf{S}$$

> ### 3. Reaction Energetics $f(T)$
> - $\Delta H_{\text{rxn}}(T)$ (Heat of reaction)
> - $\Delta S_{\text{rxn}}(T)$ (Reaction entropy)
> - $\Delta G_{\text{rxn}}(T) \;\longrightarrow\; K_{\text{eq}}(T)$ (Van 't Hoff)

$$\Big\downarrow \text{\small BEP Relations + Eyring-Polanyi}$$

> ### 4. Kinetic Rate Constants $f(T)$
> - **Forward:** $\displaystyle k_f(T) = \frac{k_B T}{h} \exp\left(-\frac{\Delta G^\ddagger_f(T)}{RT}\right)$
> - **Reverse:** $\displaystyle k_r(T) = \frac{k_f(T)}{K_{\text{eq}}(T)}$ *(Microscopic Reversibility)*

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.integrate import solve_ivp
from pathlib import Path

print('Imports OK')

Imports OK


## 2. Input data

**1. Level 1: Atomistic Data Contract (MACE-OMol / DFT)**

For every chemical species $i$ in the active library ($N_s$ species), the atomistic engine yields five physical descriptors:

| Parameter | Symbol | Units | Physical Origin |
| :--- | :--- | :--- | :--- |
| **0 K Potential Energy** | $E_{0\text{K},i}$ | $\text{eV}$ (or $\text{Hartree}$) | Minimum of the PES at relaxed equilibrium geometry |
| **Normal Mode Frequencies** | $\{\nu_{k,i}\}_{k=1}^{3N-6}$ | $\text{cm}^{-1}$ | Eigenvalues of mass-weighted Hessian matrix $\mathbf{H}_m$ |
| **Moments of Inertia** | $I_{A,i}, I_{B,i}, I_{C,i}$ | $\text{kg}\cdot\text{m}^2$ (or $\text{amu}\cdot\text{\AA}^2$) | Principal axes of inertia tensor $\mathbf{I} = \sum_a m_a(r_a^2\mathbf{1} - \mathbf{r}_a \otimes \mathbf{r}_a)$ |
| **Molecular Mass & Symmetry** | $M_i, \sigma_{\text{rot},i}$ | $\text{g/mol}$, dimensionless | Total mass $\sum m_a$ and rotational symmetry number |
| **Solvation Energy in EC** | $\Delta E_{\text{solv},i}$ | $\text{eV}$ (or $\text{kcal/mol}$) | MD potential-energy difference or implicit SMD cavity energy |

In [ ]:
# ---------------------------------------------------------------
# Free energies (Hartree) from B3LYP-D3 — same as thermo_H2O.ipynb
# ---------------------------------------------------------------
species_list = ['TMSPA','BMSPA','MMSPA','H3PO4','H2O','TMSOH',
                'siloxyl','silicone','CH4','EC','VC','EO','VO',
                'CO2','EG','VG','H2CO3','Ethane','Ethene','Ethyne',
                'H2','O2','TMSOEG','TMSOVG','TMSOCH3','TMS','TMSOdiEG']

freenergies_B3 = [
    -1870.111472, -1461.455271, -1052.799832, -644.143636, -76.417859,
    -485.067667,  -893.719314, -1338.343345,  -40.491921, -342.378695,
    -341.173421,  -153.761644,  -152.472764,  -188.607763, -230.204618,
    -229.002199,  -265.011021,   -79.780120,   -78.563543,  -77.324316,
      -1.175183,  -150.350258,  -638.851220,  -637.649600, -524.343323,
    -449.098270,  -792.639162
]

# Vibrational frequencies (cm^-1)
nu_I = 

# Moments of intertia (?)
  # I_A, I_B, I_C are three diferent matrix or just one I_i?
I = 

# Molecular masses (g/mol)
MW_I =

# ...  continue


# -------------------------------------------------------------------------

print('DFT Free energies imported OK')

HAR2EV = 27.2114  # Hartree to eV
EV2KCAL = 23.0605 # eV to kcal/mol [not used]

G_B3 = {s: g * HAR2EV for s, g in zip(species_list, freenergies_B3)}

print('DFT Free energies converted to eV OK')

DFT Free energies imported OK
DFT Free energies converted to eV OK


### 2.3. Low-Frequency Quasi-RRHO Correction

Raw DFT vibrational freq. bellow 100 cm^-1 correspond to internal rotations. 

$\nu_i \rightarrow 0 \rightarrow S_{vib} \rightarrow \inf$

### 2.2. Solvation estimates

In [9]:
# no finished run just falls back to the PCM-like estimate below, loudly.

SOLV_FALLBACK = {
    # rough PCM estimates (eV): polar/ionic species stabilised more
    'H3PO4': -0.45, 'H2O': -0.25, 'TMSOH': -1.48,
    'MMSPA': -0.30, 'BMSPA': -0.22, 'TMSPA': -1.715,
    'EC':    -0.18, 'VC':   -0.17, 'CO2':  -0.08,
    'TMSOEG':-0.20, 'TMSOdiEG': -0.22,
    'siloxyl': -0.12, 'silicone': -0.10,
    'EG': -0.60, 'VG': -0.28,
}

print('Solvation fallback corrections imported OK')

Solvation fallback corrections imported OK


In [10]:
def load_fresh_solvation_corrections(workdir: Path) -> dict:
    """dE_solv_eV from every finished solvation_runs/<name>/result.json --
    the full-precision, current-pipeline value, matching what analyze_results
    reports for the same species."""
    fresh = {}
    for f in sorted(workdir.glob('*/result.json')):
        res = json.loads(f.read_text())
        fresh[res['name']] = res['dE_solv_eV']
    return fresh

G_solv = dict(SOLV_FALLBACK)  # base: rough PCM estimate for everything

workdir = Path('solvation_runs')
fresh = load_fresh_solvation_corrections(workdir) if workdir.exists() else {}
G_solv.update(fresh)

fallback_species = [s for s in species_list if s not in fresh]
print(f'Solvation corrections: {len(fresh)} from {workdir} (current MACE runs), '
      f'{len(fallback_species)} on PCM fallback')
if fallback_species:
    print(f'  PCM fallback (no finished solvation_runs result yet): {", ".join(fallback_species)}')



Solvation corrections: 0 from solvation_runs (current MACE runs), 27 on PCM fallback
  PCM fallback (no finished solvation_runs result yet): TMSPA, BMSPA, MMSPA, H3PO4, H2O, TMSOH, siloxyl, silicone, CH4, EC, VC, EO, VO, CO2, EG, VG, H2CO3, Ethane, Ethene, Ethyne, H2, O2, TMSOEG, TMSOVG, TMSOCH3, TMS, TMSOdiEG


### 2.3. Free energy change in solution

In [13]:
# Solution-phase free energy from G_B3 and G_solv, in eV
def G_sol(species):
    """Solvation-corrected free energy in eV."""
    return G_B3.get(species, 0.0) + G_solv.get(species, 0.0)

# Reaction free-energy in solution 
def dG_rxn(reactants, products):
    """ΔG_rxn in eV (solution phase)."""
    return sum(G_sol(p) for p in products) - sum(G_sol(r) for r in reactants)

## 3. Reaction network

### 3.1. Rate constants 

$k=\frac{k_bT}{h}exp(-\frac{\Delta_G}{k_BT})$

In [14]:
# ---------------------------------------------------------------
# BEP / Eyring rate constant calculator
# ---------------------------------------------------------------
KB  = 8.617e-5   # eV/K  (Boltzmann)
kBh = 6.25e12    # kB*T/h at 298 K  (s⁻¹)
R   = 8.314e-3   # kJ/mol/K

def rate_constants(dG_rxn_eV, T=298.15, E0=0.8, alpha=0.5):
    """
    BEP barriers and Eyring rate constants.
    E0 : intrinsic barrier (eV)  — primary tuning parameter
    alpha : Brønsted coefficient (0–1)
    Returns k_f, k_r  (s⁻¹  for 1st order, or M⁻¹s⁻¹ / M⁻²s⁻¹ for higher order)
    """
    kT = KB * T
    dG_f = max(E0, E0 + alpha * dG_rxn_eV)   # forward barrier
    dG_r = dG_f - dG_rxn_eV                  # reverse barrier = f - rxn
    prefactor = kBh * (T / 298.15)
    k_f = prefactor * np.exp(-dG_f / kT)
    k_r = prefactor * np.exp(-dG_r / kT)
    return k_f, k_r

### 3.X. TST / Eyring

(Vlachos et al.) never calculate $k_f$ and $k_b$ independently from two independent DFT barriers -> sum errors (violates 1st and 2nd thermo laws).

Always calculate direct and eq. constants, derivating the inverse.



In [19]:
# Constantes físicas en unidades consistentes (eV y s)
KB_EV = 8.617333e-5   # eV / K
H_EV_S = 4.135667e-15 # eV · s
KB_H = KB_EV / H_EV_S # ~ 2.0836e10 s^-1 K^-1 (6.212e12 s^-1 a 298.15 K)

def calculate_rate_constants(dG_rxn_eV, T=298.15, E0=0.80, alpha=0.50, model='BEP'):
    """
    Calculates k_f and k_b while ensuring thermodynamic consistency
    and microscopic reversibility.

    Parameters:
    - dG_rxn_eV: Reaction free energy in solution (eV)
    - T: Absolute temperature (K)
    - E0: Intrinsic barrier (eV)
    - alpha: Transfer coefficient / Brønsted coefficient (0 to 1)
    """
    kT = KB_EV * T
    
    # 1. Barrera de activación directa dG_if
    if model == 'BEP':
        dG_if = max(E0, E0 + alpha * dG_rxn_eV)
    elif model == 'Marcus':
        dG_if = E0 * (1.0 + dG_rxn_eV / (4.0 * E0))**2
    else:
        raise ValueError(f"Modelo desconocido: {model}")
        
    # 2. Constante directa (Eyring)
    prefactor = KB_H * T  # s^-1
    k_f = prefactor * np.exp(-dG_if / kT)
    
    # 3. Constante de equilibrio termodinámica
    K_eq = np.exp(-dG_rxn_eV / kT)
    
    # 4. Reversibilidad microscópica estricta (Mhadeshwar & Vlachos eq. 3)
    k_b = k_f / K_eq
    
    return k_f, k_b

In [16]:
# Definición de la red (Reacción: reactivos -> productos)
REACTIONS = [
    ('R1', ['TMSPA', 'H2O'],        ['BMSPA', 'TMSOH']),
    ('R2', ['BMSPA', 'H2O'],        ['MMSPA', 'TMSOH']),
    ('R3', ['MMSPA', 'H2O'],        ['H3PO4', 'TMSOH']),
    ('R4', ['TMSOH', 'TMSOH'],      ['siloxyl', 'H2O']),
    ('R5', ['TMSPA', 'TMSOH'],      ['BMSPA', 'siloxyl']),
    ('R6', ['BMSPA', 'TMSOH'],      ['MMSPA', 'siloxyl']),
    ('R7', ['MMSPA', 'TMSOH'],      ['H3PO4', 'siloxyl']),
    ('R8', ['EC', 'TMSOH'],         ['TMSOEG', 'CO2']),
    ('R9', ['EC', 'TMSOEG'],        ['TMSOdiEG', 'CO2'])
]

tracked_species = [
    'TMSPA', 'BMSPA', 'MMSPA', 'H3PO4', 'TMSOH', 
    'siloxyl', 'H2O', 'EC', 'TMSOEG', 'TMSOdiEG', 'CO2'
]
n_spec = len(tracked_species)
n_rxn = len(REACTIONS)
spec_idx = {s: i for i, s in enumerate(tracked_species)}

# Construcción de la matriz estequiométrica S (n_species x n_reactions)
S = np.zeros((n_spec, n_rxn))
for j, (label, react, prod) in enumerate(REACTIONS):
    for r in react:
        if r in spec_idx:
            S[spec_idx[r], j] -= 1.0
    for p in prod:
        if p in spec_idx:
            S[spec_idx[p], j] += 1.0

print(f"Matriz estequiométrica S construida con forma: {S.shape}")
print(S)

Matriz estequiométrica S construida con forma: (11, 9)
[[-1.  0.  0.  0. -1.  0.  0.  0.  0.]
 [ 1. -1.  0.  0.  1. -1.  0.  0.  0.]
 [ 0.  1. -1.  0.  0.  1. -1.  0.  0.]
 [ 0.  0.  1.  0.  0.  0.  1.  0.  0.]
 [ 1.  1.  1. -2. -1. -1. -1. -1.  0.]
 [ 0.  0.  0.  1.  1.  1.  1.  0.  0.]
 [-1. -1. -1.  1.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0. -1. -1.]
 [ 0.  0.  0.  0.  0.  0.  0.  1. -1.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  1.]
 [ 0.  0.  0.  0.  0.  0.  0.  1.  1.]]


In [17]:
def compute_reaction_fluxes(C, k_f_vec, k_b_vec):
    """
    Calcula el vector de velocidades netas de reacción r (mol / L·s)
    según la ley de acción de masas.
    """
    r = np.zeros(n_rxn)
    for j, (label, react, prod) in enumerate(REACTIONS):
        # Término directo
        flux_f = k_f_vec[j]
        for r_name in react:
            if r_name in spec_idx:
                flux_f *= max(0.0, C[spec_idx[r_name]])
                
        # Término inverso
        flux_b = k_b_vec[j]
        for p_name in prod:
            if p_name in spec_idx:
                flux_b *= max(0.0, C[spec_idx[p_name]])
                
        r[j] = flux_f - flux_b
    return r

def tank_rhs(t, C, k_f_vec, k_b_vec, S):
    """
    Balance de materia del Reactor Batch ideal cerrado (Tank):
    dC/dt = S · r(C, T)
    """
    r = compute_reaction_fluxes(C, k_f_vec, k_b_vec)
    return S @ r

In [20]:
from scipy.integrate import solve_ivp

T_sim = 298.15 # K

# 1. Evaluar dG_rxn para cada reacción en solución y sus constantes
k_f_list = []
k_b_list = []

for label, react, prod in REACTIONS:
    # dG_rxn = sum(G_sol(prod)) - sum(G_sol(react))
    dg = sum(G_B3[p] + SOLV_FALLBACK.get(p, 0.0) for p in prod) - \
         sum(G_B3[r] + SOLV_FALLBACK.get(r, 0.0) for r in react)
         
    # Asignar E0 según el modelo (ej. E0 = 0.8 eV para Model 1/2 inicial)
    kf, kb = calculate_rate_constants(dg, T=T_sim, E0=0.80, alpha=0.50, model='BEP')
    k_f_list.append(kf)
    k_b_list.append(kb)

k_f_vec = np.array(k_f_list)
k_b_vec = np.array(k_b_list)

# 2. Condiciones iniciales del electrolito (Molares)
C0 = np.zeros(n_spec)
C0[spec_idx['TMSPA']] = 0.050   # 50 mM aditivo
C0[spec_idx['H2O']]   = 0.005   # 5 mM (~100 ppm H2O inicial)
C0[spec_idx['EC']]    = 10.0    # Disolvente mayoritario

# 3. Intervalo de tiempo (ej. 7 días)
t_span = (0.0, 7 * 24 * 3600)
t_eval = np.logspace(0, np.log10(t_span[1]), 500)

# 4. Integración del sistema rígido (Radau IIA)
solution = solve_ivp(
    tank_rhs, t_span, C0, 
    args=(k_f_vec, k_b_vec, S),
    method='Radau', 
    t_eval=t_eval,
    rtol=1e-7, atol=1e-10
)

print(f"Simulación finalizada exitosamente: {solution.success}")

Simulación finalizada exitosamente: True


## 4. Models
### Model 1: Gas-Phase Baseline (Vacuum)

### Model 2: Implicir Solvation